In [5]:
import os
os.getcwd()


'/home/onyxia/work/prediction-musee/notebook'

In [6]:
import pandas as pd
df = pd.read_csv("../output/df_modele_musees.csv")
df.columns

Index(['id_patrimostat', 'id_museofile', 'dateappellation', 'ferme',
       'anneefermeture', 'ville', 'codeInseeCommune', 'annee', 'payant',
       'gratuit', 'total', 'individuel', 'scolaires', 'groupes_hors_scolaires',
       'moins_18_ans_hors_scolaires', '_18_25_ans', 'part_gratuit',
       'part_scolaires', 'part_individuels', 'nom_officiel', 'region',
       'departement', 'categorie', 'domaine_thematique', 'annee_creation',
       'latitude', 'longitude', 'total_frequentation', 'age_musee',
       'age_musee_missing', 'total_t_1', 'croissance_total', 'has_excel',
       'est_idf'],
      dtype='object')

In [9]:
# ==============================
# MODELE ML - FREQUENTATION MUSEES
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1) Charger les données
df = pd.read_csv("../output/df_modele_musees.csv")

# 2) Définir la target
target = "total_frequentation"

# 3) Supprimer les lignes où la target est manquante
df = df.dropna(subset=[target])

# 4) Colonnes à exclure (data leakage)
leakage_cols = [
    "total_frequentation",
    "total",
    "total_t_1",
    "croissance_total"
]

X = df.drop(columns=leakage_cols)
y = df[target]

# 5) Garder colonnes exploitables
X = X.select_dtypes(include=["number", "object", "bool"])

# 6) Encoder les catégories
X = pd.get_dummies(X, drop_first=True)

# 7) Remplacer les NaN restants dans X
X = X.fillna(0)

# 8) Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 9) Modèle
model = RandomForestRegressor(
    n_estimators=20,
    random_state=42
)

# 10) Entraînement
model.fit(X_train, y_train)

# 11) Prédictions
y_pred = model.predict(X_test)

# 12) Résultats
print("MAE :", round(mean_absolute_error(y_test, y_pred), 2))
print("R² :", round(r2_score(y_test, y_pred), 3))

# 13) Aperçu réel vs prédit
display(pd.DataFrame({
    "Frequentation_reelle": y_test.values[:10],
    "Frequentation_predite": y_pred[:10]
}))


MAE : 2403.74
R² : 0.986


,Frequentation_reelle,Frequentation_predite
0,10935.0,10919.00
1,31566.0,31396.75
2,23305.0,23347.60
3,0.0,0.00
4,72568.0,72905.65
5,16175.0,16260.30
6,759.0,768.55
7,786.0,758.65
8,7310.0,7411.40
9,1099.0,1089.65
